In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

print(v1.shape)

(384,)


In [9]:
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [10]:
similarity = model.similarity(v1, dv)
print(similarity)

tensor([[0.3233]])


In [11]:
v1.dot(dv)

np.float32(0.323324)

In [12]:
# TEST
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings)
# [3, 384]

[[ 0.01919576  0.12008537  0.15959829 ... -0.00536282 -0.08109502
   0.05021341]
 [-0.01869034  0.0415187   0.07431547 ...  0.004866   -0.06190434
   0.03187511]
 [ 0.136502    0.08227322 -0.02526164 ...  0.08762049  0.03045843
  -0.01075752]]


In [13]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)
v2

array([-2.35677883e-02,  7.85555765e-02,  2.33792625e-02, -6.72475435e-03,
       -2.58874726e-02,  1.18256640e-02,  5.29527431e-04,  9.57366079e-03,
        6.23929128e-02,  2.91427094e-02, -4.18229997e-02, -6.62149265e-02,
       -2.76613347e-02,  5.41518740e-02, -1.92781650e-02, -2.46782135e-02,
       -2.89787864e-03,  8.40462074e-02,  1.03637632e-02, -6.35393988e-03,
       -1.20814100e-01,  1.80162862e-02, -5.70900701e-02, -7.92970881e-02,
       -9.25547816e-03, -3.24046090e-02, -1.95662733e-02, -2.65410803e-02,
       -9.41327959e-03,  4.31234725e-02, -2.08750162e-02, -7.86434039e-02,
        9.19373557e-02, -3.75263649e-03,  5.13071194e-02, -5.73638361e-03,
        5.61954305e-02,  3.73761216e-03, -6.88236579e-02, -4.57000434e-02,
        2.83076130e-02, -3.49764675e-02, -5.55221282e-04, -3.09770070e-02,
        4.34274971e-02, -4.48056757e-02,  6.84109777e-02,  4.14158509e-04,
        5.82685843e-02,  6.66982681e-02, -8.95875692e-02, -2.69941594e-02,
        1.06746368e-02, -

In [14]:
v2.dot(dv)

np.float32(0.019730454)

In [15]:
from ingest import load_faq_data

documents = load_faq_data()
len(documents)

1350

In [16]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

len(texts)

L = [3,4,5,6,74,8]

M = L[2:len(L)]
M

[5, 6, 74, 8]

In [17]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

In [18]:
import numpy as np
X = np.array(vectors)

In [19]:
query = "Can I still join the course after the start date?"

v_query = model.encode(query)

In [20]:
scores = X.dot(v_query)

In [21]:
scores = [X[i].dot(v_query) for i in range(len(X))]

In [22]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [23]:
texts[idx]

"Course: Can I still join the course after the start date? Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."

In [24]:
top5 = np.argsort(scores)[-5:]
top5

array([  7, 538, 907, 625,   2])

In [25]:
top5 = top5[::-1]

In [26]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579372
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192131
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related

In [27]:
#Vector Search with minsearch
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [28]:
query = "I just discovered the course. Can I still join it"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [29]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [30]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [31]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [32]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [33]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [34]:
from rag_helper import RAGBAse

assistant = RAGBAse(
    index=index,
    llm_client=openai_client
)

In [35]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want a certificate, make sure you submit your project while submissions are still being accepted.'

In [36]:
class RAGVector(RAGBAse):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [37]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client
)

In [38]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join even if the program has already begun. You can start learning and submitting homework while the submission form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [39]:
# Vector Search with sqlitesearch
'''
What we've done so far is exact nearest neighbor (NN) search. We score the query against every document and pick the top ones. It always finds the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search takes a shortcut. Instead of comparing against everything, it first narrows down to a region of likely matches. Then it scores only within that region. It may miss the absolute best match, but the results are still good and it's much faster.
'''

"\nWhat we've done so far is exact nearest neighbor (NN) search. We score the query against every document and pick the top ones. It always finds the true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of comparing against everything, it first narrows down to a region of likely matches. Then it scores only within that region. It may miss the absolute best match, but the results are still good and it's much faster.\n"

In [40]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [41]:
vs_index.fit(vectors, documents)

In [45]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
print(results)

[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}, {'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Relat

In [48]:
results = vs_index.search(
    query_vector, 
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [49]:
vs_index.close()